In [1]:
from langchain.vectorstores.chroma import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama
from langchain_community.embeddings.ollama import OllamaEmbeddings

In [2]:
def get_embedding_function():
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    return embeddings

In [3]:
import os
from langchain.document_loaders.pdf import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema.document import Document
from langchain.vectorstores.chroma import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama
from langchain_community.document_loaders import PyPDFLoader

In [4]:
CHROMA_PATH = "chroma"
DATA_PATH = r"C:\Users\LENOVO/Downloads\catan_base_rules_2020_200707.pdf"

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=80,
        length_function=len,
        is_separator_regex=False,
    )

In [10]:
def load_documents():
    document_loader = PyPDFLoader(DATA_PATH)
    return document_loader.load()

In [11]:
documents = load_documents()

#### Chunking strategy #1
#### Fixed Size Chunking - Based on number of tokens

In [13]:
def split_documents(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=80,
        length_function=len,
        is_separator_regex=False,
    )
    return text_splitter.split_documents(documents)

In [14]:
def add_to_chroma(chunks: list[Document]):
    # Load the existing database.
    db = Chroma(
        persist_directory=CHROMA_PATH, embedding_function=get_embedding_function()
    )

    # Calculate Page IDs.
    chunks_with_ids = calculate_chunk_ids(chunks)

    # Add or Update the documents.
    existing_items = db.get(include=[])  # IDs are always included by default
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    # Only add documents that don't exist in the DB.
    new_chunks = []
    for chunk in chunks_with_ids:
        if chunk.metadata["id"] not in existing_ids:
            new_chunks.append(chunk)

    if len(new_chunks):
        print(f"Adding new documents: {len(new_chunks)}")
        new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
        db.add_documents(new_chunks, ids=new_chunk_ids)
        db.persist()
    else:
        print("No new documents to add")

In [15]:
def calculate_chunk_ids(chunks):

    last_page_id = None
    current_chunk_index = 0

    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        current_page_id = f"{source}:{page}"

        # If the page ID is the same as the last one, increment the index.
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0

        # Calculate the chunk ID.
        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id

        # Add it to the page meta-data.
        chunk.metadata["id"] = chunk_id

    return chunks

In [16]:
# Create (or update) the data store.

chunks = split_documents(documents)
add_to_chroma(chunks)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5988\552428272.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5988\535170893.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


Number of existing documents in DB: 0
Adding new documents: 81


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5988\535170893.py:25: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [24]:
PROMPT_TEMPLATE = """
You are a helpful assistant. Answer the question based only on the following context:

{context}

---

Answer the question based on the above context: {question}
"""


In [26]:
def query_rag(query_text: str):
    # Prepare the DB.
    embedding_function = get_embedding_function()
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

    # Search the DB.
    results = db.similarity_search_with_score(query_text, k=5)

    context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
    prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
    prompt = prompt_template.format(context=context_text, question=query_text)
    # print(prompt)

    model = Ollama(model="gemma3:1b")
    response_text = model.invoke(prompt)

    sources = [doc.metadata.get("id", None) for doc, _score in results]
    formatted_response = f"Response: {response_text}\nSources: {sources}"
    print(formatted_response)
    return response_text


In [30]:
print(query_rag("what happens when a player rolls a 7? Be elaborate"))

Response: Okay, let’s break down what happens when a player rolls a “7” in the game based on the provided context.

When a player rolls a “7,” **no resources are awarded.** Instead, the following occurs:

1.  **Everyone with more than 7 resources** must select half of their resource cards and return them to the bank.
2.  **The robber is moved immediately to the number token of any other terrain hex or to the desert hex.** This is the immediate action.
3.  **The player then steals 1 random resource card** from an opponent who has a settlement or city adjacent to the target terrain hex. The player who is robbed holds their resource cards face down.
4.  **The player then takes 1 card at random.** This ensures that the robber remains in the same location.

**In essence, the “7” represents a situation where the robber is triggered, and the game shifts into a state of potential resource redistribution.** It’s a critical event that forces a change in the game’s dynamics.


Sources: ['C:\\User

#### Chunking strategy #2
#### Semantic chunking

In [34]:
from langchain_experimental.text_splitter import SemanticChunker

In [ ]:
semantic_chunker = SemanticChunker(get_embedding_function(), breakpoint_threshold_type="percentile")
semantic_chunks = semantic_chunker.create_documents([d.page_content for d in documents])

In [ ]:
semantic_chunk_vectorstore = Chroma.from_documents(semantic_chunks, embedding=get_embedding_function())

In [ ]:
semantic_chunk_retriever = semantic_chunk_vectorstore.as_retriever(search_kwargs={"k" : 1})
semantic_chunk_retriever.invoke("What happens when a player rolls a 7?")

In [ ]:
rag_template = """\
You are a helpful assistant. Answer the question based only on the following context:

{context}

---

Answer the question based on the above context: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

semantic_rag_chain = (
    {"context" : semantic_chunk_retriever, "question" : RunnablePassthrough()}
    | rag_prompt
    | chat_model
    | StrOutputParser()
)